In [10]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure NLTK data is downloaded for text cleaning
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("--- Starting Phase 1: Data Preparation & Preprocessing ---")

# ==========================================
# 1. Configuration & Paths
# ==========================================
TRAIN_FOLDER = '/kaggle/input/datasets/suryanarayanghosh/trainmami/train/'
TEST_FOLDER = '/kaggle/input/datasets/suryanarayanghosh/testmami/test/'
TRAIN_CSV = 'train.csv'
TEST_CSV = 'test.csv'

stop_words = set(stopwords.words('english'))

# ==========================================
# 2. Helper Functions
# ==========================================
def clean_text(text):
    """Cleans text by lowercasing, removing special chars, and removing stopwords."""
    if pd.isna(text) or not isinstance(text, str): 
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    word_tokens = word_tokenize(text)
    return " ".join([w for w in word_tokens if w not in stop_words])

def prepare_data(folder_path, csv_filename, mode='train'):
    """Loads CSV, fixes image paths, cleans text, and maps labels."""
    csv_path = os.path.join(folder_path, csv_filename)
    print(f"Loading {mode} dataset from: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    # a. Drop 'chinese_labels' if it exists
    if 'chinese_labels' in df.columns:
        df = df.drop(columns=['chinese_labels'])
        
    # b. Fix image paths (Convert int to string, add .jpg if missing, and join path)
    df['image_id'] = df['image_id'].apply(
        lambda x: os.path.join(folder_path, f"{x}.jpg" if not str(x).lower().endswith(('.jpg', '.png', '.jpeg')) else str(x))
    )
    
    # c. Clean text transcriptions
    print(f"Cleaning text transcriptions for {mode} data...")
    df['transcriptions'] = df['transcriptions'].apply(clean_text)
    
    # d. Handle labels mapping (String to 1/0)
    if mode == 'test':
        # Create a dummy label for unlabelled test set so pipelines don't break
        df['misogyny'] = -1 
    else:
        # Create new column 'misogyny': 1 if misogyny, 0 if not-misogyny
        df['misogyny'] = df['indian_labels'].apply(lambda x: 1 if str(x).strip().lower() == 'misogyny' else 0)
        
    # e. Keep only the final required columns
    return df[['image_id', 'transcriptions', 'misogyny']]

# ==========================================
# 3. Execution & Splitting
# ==========================================
# Process Train Data
train_full_df = prepare_data(TRAIN_FOLDER, TRAIN_CSV, mode='train')

# Process Unlabelled Test Data
test_df = prepare_data(TEST_FOLDER, TEST_CSV, mode='test')

# Check class balance
print("\n--- Training Data Balance ---")
class_counts = train_full_df['misogyny'].value_counts()
print(class_counts)
print(f"Misogynous (1) percentage: {class_counts.get(1, 0) / len(train_full_df) * 100:.2f}%\n")

# Train-Val Split (70:30)
print("Performing 70:30 Train-Validation Split...")
train_df, val_df = train_test_split(
    train_full_df, 
    test_size=0.30, 
    random_state=SEED, 
    stratify=train_full_df['misogyny'] # Ensures balance is maintained
)

print(f"✅ Final Train Set Size: {len(train_df)}")
print(f"✅ Final Validation Set Size: {len(val_df)}")
print(f"✅ Unlabelled Test Set Size: {len(test_df)}\n")

# ==========================================
# 4. Handle Imbalance (Class Weights)
# ==========================================
print("--- Handling Imbalance (Calculating Class Weights) ---")
# Calculate class weights based ONLY on the training split to avoid data leakage
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['misogyny']),
    y=train_df['misogyny']
)

# Convert to a PyTorch tensor (we will pass this to the loss function in Phase 2)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
print(f"✅ Calculated Class Weights [Class 0, Class 1]: {class_weights_tensor}")

print("\nSnippet of prepared training data:")
pd.set_option('display.max_colwidth', None)
print(train_df.head(2))

print("\n--- Phase 1 Complete ---")

--- Starting Phase 1: Data Preparation & Preprocessing ---
Loading train dataset from: /kaggle/input/datasets/suryanarayanghosh/trainmami/train/train.csv
Cleaning text transcriptions for train data...
Loading test dataset from: /kaggle/input/datasets/suryanarayanghosh/testmami/test/test.csv
Cleaning text transcriptions for test data...

--- Training Data Balance ---
misogyny
0    6557
1    2838
Name: count, dtype: int64
Misogynous (1) percentage: 30.21%

Performing 70:30 Train-Validation Split...
✅ Final Train Set Size: 6576
✅ Final Validation Set Size: 2819
✅ Unlabelled Test Set Size: 1000

--- Handling Imbalance (Calculating Class Weights) ---
✅ Calculated Class Weights [Class 0, Class 1]: tensor([0.7163, 1.6556])

Snippet of prepared training data:
                                                               image_id  \
4326  /kaggle/input/datasets/suryanarayanghosh/trainmami/train/6572.jpg   
9370  /kaggle/input/datasets/suryanarayanghosh/trainmami/train/7709.jpg   

            

In [11]:
from sklearn.utils import resample

print("--- Starting Phase 1.1: Data Balancing (Oversampling) ---")

# 1. Separate majority (0) and minority (1) classes from the training set
df_majority = train_df[train_df['misogyny'] == 0]
df_minority = train_df[train_df['misogyny'] == 1]

print(f"Original Majority class (0) size: {len(df_majority)}")
print(f"Original Minority class (1) size: {len(df_minority)}")

# 2. Upsample the minority class
df_minority_upsampled = resample(
    df_minority, 
    replace=True,                  # Sample with replacement (duplicate rows)
    n_samples=len(df_majority),    # Match the length of the majority class
    random_state=SEED              # Ensure reproducibility
)

# 3. Combine majority class with the upsampled minority class
train_df_balanced = pd.concat([df_majority, df_minority_upsampled])

# 4. Shuffle the newly combined dataset so classes are mixed perfectly
train_df = train_df_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)

# 5. Verify the new balance
print("\n✅ New Training Data Balance after Oversampling:")
new_class_counts = train_df['misogyny'].value_counts()
print(new_class_counts)
print(f"Misogynous (1) percentage: {new_class_counts.get(1, 0) / len(train_df) * 100:.2f}%\n")

print(f"✅ Updated Final Train Set Size: {len(train_df)}")
print(f"✅ Validation Set Size remains unchanged: {len(val_df)}")

print("\n--- Phase 1.1 Complete ---")

--- Starting Phase 1.1: Data Balancing (Oversampling) ---
Original Majority class (0) size: 4590
Original Minority class (1) size: 1986

✅ New Training Data Balance after Oversampling:
misogyny
1    4590
0    4590
Name: count, dtype: int64
Misogynous (1) percentage: 50.00%

✅ Updated Final Train Set Size: 9180
✅ Validation Set Size remains unchanged: 2819

--- Phase 1.1 Complete ---


In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from PIL import Image
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings("ignore")

print("--- Starting Phase 2: Unimodal Training ---")

# ==========================================
# 1. Hyperparameters & Setup
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
MAX_LEN = 128

# Create directory in Kaggle's working folder to save our models
SAVE_DIR = '/kaggle/working/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# 2. PyTorch Datasets
# ==========================================
class TextDataset(Dataset):
    def __init__(self, df, tokenizer_name):
        self.texts = df['transcriptions'].reset_index(drop=True)
        self.labels = df['misogyny'].reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class ImageDataset(Dataset):
    def __init__(self, df, transform=None, processor_name=None):
        self.img_paths = df['image_id'].reset_index(drop=True)
        self.labels = df['misogyny'].reset_index(drop=True)
        self.transform = transform
        self.processor = AutoImageProcessor.from_pretrained(processor_name) if processor_name else None

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.img_paths[idx]).convert('RGB')
        except Exception as e:
            img = Image.new('RGB', (224, 224), color='black') # Fallback for corrupted images
            
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        if self.processor:
            pixel_values = self.processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
            return {'image': pixel_values, 'label': label}
        else:
            if self.transform: img = self.transform(img)
            return {'image': img, 'label': label}

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 3. Model Architectures
# ==========================================
class TextModel(nn.Module):
    def __init__(self, model_name):
        super(TextModel, self).__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        return self.classifier(pooled)

class ResNetModel(nn.Module):
    def __init__(self):
        super(ResNetModel, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1]) # Keep everything except final FC
        self.classifier = nn.Linear(resnet.fc.in_features, 2)

    def forward(self, x):
        features = self.backbone(x)
        features = torch.flatten(features, 1)
        return self.classifier(features)

class ViTModelCustom(nn.Module):
    def __init__(self, model_name):
        super(ViTModelCustom, self).__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        pooled = outputs.pooler_output if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        return self.classifier(pooled)

# ==========================================
# 4. Universal Training Engine (Updated Save Logic)
# ==========================================
def train_and_evaluate(model, train_loader, val_loader, model_name, is_text=True, is_hf_vision=False):
    print(f"\n🚀 Training {model_name}...")
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    best_macro_f1 = 0.0
    safe_name = model_name.replace('/', '_')

    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
            optimizer.zero_grad()
            
            labels = batch['label'].to(DEVICE)
            if is_text:
                outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            else:
                outputs = model(batch['image'].to(DEVICE))
                
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- Validation ---
        model.eval()
        val_preds, val_labels, val_probs = [], [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
                labels = batch['label'].numpy()
                if is_text:
                    outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                else:
                    outputs = model(batch['image'].to(DEVICE))
                
                probs = torch.softmax(outputs, dim=1).cpu().numpy()[:, 1]
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                
                val_preds.extend(preds)
                val_labels.extend(labels)
                val_probs.extend(probs)

        # Calculate Metrics
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        val_auc = roc_auc_score(val_labels, val_probs)

        print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Val Macro F1: {val_f1:.4f}")

        # --- SMART SAVING FOR FUSION ---
        if val_f1 > best_macro_f1:
            best_macro_f1 = val_f1
            print(f"⭐ Saving new best backbone for {model_name}...")
            
            if is_text or is_hf_vision:
                # Save HuggingFace backbone and config so it can be loaded with AutoModel in Phase 3
                backbone_dir = os.path.join(SAVE_DIR, f"{safe_name}_backbone")
                model.backbone.save_pretrained(backbone_dir)
                
                # Save tokenizer/processor along with it
                if is_text:
                    AutoTokenizer.from_pretrained(model_name).save_pretrained(backbone_dir)
                else:
                    AutoImageProcessor.from_pretrained(model_name).save_pretrained(backbone_dir)
            else:
                # For standard PyTorch ResNet, just save the backbone state_dict
                torch.save(model.backbone.state_dict(), os.path.join(SAVE_DIR, f"{safe_name}_backbone.pth"))
            
    # Clean up
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ Finished {model_name}. Best Macro F1: {best_macro_f1:.4f}\n")

# ==========================================
# 5. Execution Pipeline
# ==========================================

# A. Train Text Models
text_models = ['roberta-base', 'google/muril-base-cased']
for tm in text_models:
    train_loader = DataLoader(TextDataset(train_df, tm), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TextDataset(val_df, tm), batch_size=BATCH_SIZE, shuffle=False)
    
    model = TextModel(tm)
    train_and_evaluate(model, train_loader, val_loader, tm, is_text=True, is_hf_vision=False)

# B. Train Image Models
# 1. ResNet50
train_loader = DataLoader(ImageDataset(train_df, transform=resnet_transform), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ImageDataset(val_df, transform=resnet_transform), batch_size=BATCH_SIZE, shuffle=False)
resnet = ResNetModel()
train_and_evaluate(resnet, train_loader, val_loader, "resnet50", is_text=False, is_hf_vision=False)

# 2. ViT
vit_name = 'google/vit-base-patch16-224-in21k'
train_loader = DataLoader(ImageDataset(train_df, processor_name=vit_name), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ImageDataset(val_df, processor_name=vit_name), batch_size=BATCH_SIZE, shuffle=False)
vit = ViTModelCustom(vit_name)
train_and_evaluate(vit, train_loader, val_loader, vit_name, is_text=False, is_hf_vision=True)

print("\n--- Phase 2 Complete. Fine-tuned Backbones prepped for Multimodal Fusion! ---")

--- Starting Phase 2: Unimodal Training ---
Using Device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Epoch 3/3 [Val]: 100%|██████████| 177/177 [01:02<00:00,  2.82it/s]

Epoch 3 | Loss: 0.0683 | Val Acc: 0.7403 | Val AUC: 0.7102 | Val Macro F1: 0.6471
⭐ Saving new best backbone for google/vit-base-patch16-224-in21k...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


✅ Finished google/vit-base-patch16-224-in21k. Best Macro F1: 0.6471


--- Phase 2 Complete. Fine-tuned Backbones prepped for Multimodal Fusion! ---


In [7]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from PIL import Image
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings("ignore")

print("--- Starting Phase 3: MuRIL + ResNet50 Fusion ---")

# ==========================================
# 1. Setup & Config
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

BATCH_SIZE = 32
EPOCHS = 3 
LR = 1e-3
MAX_LEN = 128
SAVE_DIR = '/kaggle/working/saved_models'

# Point strictly to the exact nested directories shown in your screenshot
MURIL_PATH = os.path.join(SAVE_DIR, 'muril_backbone', 'muril')
RESNET_DIR = os.path.join(SAVE_DIR, 'resnet_model')

fusion_strategies = ['concat', 'avg_pool', 'element_wise', 'gated']

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 2. Multimodal Dataset
# ==========================================
class MurilResnetDataset(Dataset):
    def __init__(self, df, tokenizer_path):
        self.df = df.reset_index(drop=True)
        # Using the corrected local path
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.transform = resnet_transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        # Text
        text = str(self.df.loc[idx, 'transcriptions'])
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        
        # Image
        try:
            img = Image.open(self.df.loc[idx, 'image_id']).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), color='black')
            
        image_tensor = self.transform(img)
        label = torch.tensor(self.df.loc[idx, 'misogyny'], dtype=torch.long)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'image': image_tensor,
            'label': label
        }

# ==========================================
# 3. Fusion Architecture
# ==========================================
class MurilResnetFusionNet(nn.Module):
    def __init__(self, text_bb, image_bb, fusion_type):
        super().__init__()
        self.text_bb = text_bb
        self.image_bb = image_bb
        self.fusion_type = fusion_type
        
        # Freeze Backbones
        for p in self.text_bb.parameters(): p.requires_grad = False
        for p in self.image_bb.parameters(): p.requires_grad = False
            
        # Projection Layers (Forces MuRIL 768 and ResNet 2048 into the same 512 dimension)
        self.common_dim = 512
        self.t_proj = nn.Linear(768, self.common_dim)
        self.v_proj = nn.Linear(2048, self.common_dim)
        
        # Determine dimension after fusion
        if fusion_type == 'concat':
            fused_dim = self.common_dim * 2
        elif fusion_type in ['avg_pool', 'element_wise']:
            fused_dim = self.common_dim
        elif fusion_type == 'gated':
            fused_dim = self.common_dim
            self.gate = nn.Linear(self.common_dim * 2, self.common_dim * 2)
            self.sigmoid = nn.Sigmoid()
            
        # Final Classifier
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, images):
        # 1. Extract raw features
        t_out = self.text_bb(input_ids=input_ids, attention_mask=attention_mask)
        t_feat = t_out.pooler_output if hasattr(t_out, 'pooler_output') and t_out.pooler_output is not None else t_out.last_hidden_state[:, 0, :]
        
        v_feat = self.image_bb(images)
        if v_feat.dim() > 2: v_feat = torch.flatten(v_feat, 1)
            
        # 2. Project to Common Dimension (512)
        t_p = self.t_proj(t_feat)
        v_p = self.v_proj(v_feat)
        
        # 3. Apply Fusion Strategy
        if self.fusion_type == 'concat':
            fused = torch.cat((t_p, v_p), dim=1)
        elif self.fusion_type == 'avg_pool':
            fused = (t_p + v_p) / 2.0
        elif self.fusion_type == 'element_wise':
            fused = t_p * v_p
        elif self.fusion_type == 'gated':
            concat_feat = torch.cat((t_p, v_p), dim=1)
            gates = self.sigmoid(self.gate(concat_feat))
            gate_t = gates[:, :self.common_dim]
            gate_v = gates[:, self.common_dim:]
            fused = (gate_t * t_p) + (gate_v * v_p)
            
        # 4. Classify
        return self.classifier(fused)

# ==========================================
# 4. Backbone Loading Helper
# ==========================================
def get_pretrained_backbones():
    print(f"Loading MuRIL weights from {MURIL_PATH}...")
    text_model = AutoModel.from_pretrained(MURIL_PATH)
    
    # Automatically find the weights file inside the ResNet folder
    weight_files = glob.glob(os.path.join(RESNET_DIR, '**/*.pth'), recursive=True) + \
                   glob.glob(os.path.join(RESNET_DIR, '**/*.bin'), recursive=True) + \
                   glob.glob(os.path.join(RESNET_DIR, '**/*.pt'), recursive=True)
                   
    if not weight_files:
        raise FileNotFoundError(f"Could not find any weights file in {RESNET_DIR}. Please check the folder contents.")
        
    resnet_weights_path = weight_files[0]
    print(f"Found ResNet weights at: {resnet_weights_path}")
    
    resnet_base = models.resnet50()
    image_model = nn.Sequential(*list(resnet_base.children())[:-1])
    
    # Load state dict safely handling potential mismatches or strict loading
    state_dict = torch.load(resnet_weights_path, map_location=DEVICE, weights_only=True)
    # If the saved model was the entire model rather than just the state dict, extract it:
    if not isinstance(state_dict, dict):
        state_dict = state_dict.state_dict()
        
    try:
        image_model.load_state_dict(state_dict, strict=False)
    except Exception as e:
        print(f"Warning during loading ResNet weights: {e}")
        
    return text_model, image_model

# ==========================================
# 5. Fusion Grid Search
# ==========================================
print("\n[Data] Preparing DataLoaders...")
train_loader = DataLoader(MurilResnetDataset(train_df, MURIL_PATH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MurilResnetDataset(val_df, MURIL_PATH), batch_size=BATCH_SIZE, shuffle=False)

global_best_f1 = 0.0
global_best_fusion = ""
best_model_path = os.path.join(SAVE_DIR, 'best_muril_resnet_fusion.pth')

for f_type in fusion_strategies:
    print(f"\n--- Testing Fusion Strategy: [{f_type}] ---")
    
    t_bb, i_bb = get_pretrained_backbones()
    model = MurilResnetFusionNet(t_bb, i_bb, f_type).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    
    best_local_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Train]", leave=False):
            optimizer.zero_grad()
            outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['image'].to(DEVICE))
            loss = criterion(outputs, batch['label'].to(DEVICE))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_preds, val_labels, val_probs = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                labels = batch['label'].numpy()
                outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['image'].to(DEVICE))
                probs = torch.softmax(outputs, dim=1).cpu().numpy()[:, 1]
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                
                val_preds.extend(preds)
                val_labels.extend(labels)
                val_probs.extend(probs)
                
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        val_auc = roc_auc_score(val_labels, val_probs)
        
        print(f"Epoch {epoch+1} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f}")
        
        # Track best model globally
        if val_f1 > global_best_f1:
            global_best_f1 = val_f1
            global_best_fusion = f_type
            torch.save({
                'fusion_type': f_type,
                'state_dict': model.state_dict()
            }, best_model_path)
            print(f"⭐ New GLOBAL BEST MODEL found and saved! (Macro F1: {global_best_f1:.4f})")
    
    # Clean up memory before next fusion strategy
    del model, t_bb, i_bb, optimizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n==============================================")
print(f"🎉 PHASE 3 FUSION SEARCH COMPLETE!")
print(f"🏆 Best Fusion Strategy: {global_best_fusion}")
print(f"🏆 Winning Macro F1: {global_best_f1:.4f}")
print(f"💾 Saved best model at: {best_model_path}")
print("==============================================")

--- Starting Phase 3: MuRIL + ResNet50 Fusion ---
Using Device: cuda

[Data] Preparing DataLoaders...

--- Testing Fusion Strategy: [concat] ---
Loading MuRIL weights from /kaggle/working/saved_models/muril_backbone/muril...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Found ResNet weights at: /kaggle/working/saved_models/resnet_model/resnet50_backbone.pth


Epoch 1 | Acc: 0.7556 | AUC: 0.7433 | F1: 0.6861
⭐ New GLOBAL BEST MODEL found and saved! (Macro F1: 0.6861)


Epoch 2 | Acc: 0.7559 | AUC: 0.7308 | F1: 0.6771


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Found ResNet weights at: /kaggle/working/saved_models/resnet_model/resnet50_backbone.pth


Epoch 1 | Acc: 0.7457 | AUC: 0.7277 | F1: 0.6618


Epoch 2 | Acc: 0.7520 | AUC: 0.7377 | F1: 0.6493


Epoch 3 | Acc: 0.7457 | AUC: 0.7431 | F1: 0.6861

🎉 PHASE 3 FUSION SEARCH COMPLETE!
🏆 Best Fusion Strategy: element_wise
🏆 Winning Macro F1: 0.6892
💾 Saved best model at: /kaggle/working/saved_models/best_muril_resnet_fusion.pth


In [4]:
# Create the destination folder
!mkdir -p /kaggle/working/saved_models

# Copy MuRIL directory (assuming it's a folder containing HuggingFace files)
!cp -r /kaggle/input/models/suryanarayanghosh/muril/pytorch/default/1 /kaggle/working/saved_models/muril_backbone

# Copy ResNet file (assuming it's a single .pth file, adjust the name if needed)
!cp -r /kaggle/input/models/suryanarayanghosh/resnet/pytorch/default/1 /kaggle/working/saved_models/resnet_model

# Verify they copied successfully
!ls /kaggle/working/saved_models

muril_backbone	resnet_model


In [6]:
!rm -rf /kaggle/working/saved_models

In [14]:
!pip install sentencepiece tiktoken

In [8]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModel
from PIL import Image
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("--- Starting Phase 4: Model Inference ---")

# ==========================================
# 1. Setup & Config
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

BATCH_SIZE = 32
MAX_LEN = 128
MODEL_PATH = '/kaggle/working/saved_models/best_muril_resnet_fusion.pth' # Adjust if your exact filename differs
SUBMISSION_PATH = 'submission.csv'

# Same transform used during training
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 2. Test Dataset Definition
# ==========================================
class MurilResnetTestDataset(Dataset):
    def __init__(self, df, tokenizer_name="google/muril-base-cased"):
        self.df = df.reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.transform = resnet_transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        # Text
        text = str(self.df.loc[idx, 'transcriptions'])
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        
        # Image
        img_path = self.df.loc[idx, 'image_id']
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), color='black') # Fallback for corrupted images
            
        image_tensor = self.transform(img)
        file_name = os.path.basename(img_path) # Extract just the filename for submission

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'image': image_tensor,
            'file_name': file_name
        }

# ==========================================
# 3. Model Architecture (Must match training)
# ==========================================
class MurilResnetFusionNet(nn.Module):
    def __init__(self, text_bb, image_bb, fusion_type):
        super().__init__()
        self.text_bb = text_bb
        self.image_bb = image_bb
        self.fusion_type = fusion_type
        
        self.common_dim = 512
        self.t_proj = nn.Linear(768, self.common_dim)
        self.v_proj = nn.Linear(2048, self.common_dim)
        
        if fusion_type == 'concat':
            fused_dim = self.common_dim * 2
        elif fusion_type in ['avg_pool', 'element_wise']:
            fused_dim = self.common_dim
        elif fusion_type == 'gated':
            fused_dim = self.common_dim
            self.gate = nn.Linear(self.common_dim * 2, self.common_dim * 2)
            self.sigmoid = nn.Sigmoid()
            
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, images):
        t_out = self.text_bb(input_ids=input_ids, attention_mask=attention_mask)
        t_feat = t_out.pooler_output if hasattr(t_out, 'pooler_output') and t_out.pooler_output is not None else t_out.last_hidden_state[:, 0, :]
        
        v_feat = self.image_bb(images)
        if v_feat.dim() > 2: v_feat = torch.flatten(v_feat, 1)
            
        t_p = self.t_proj(t_feat)
        v_p = self.v_proj(v_feat)
        
        if self.fusion_type == 'concat':
            fused = torch.cat((t_p, v_p), dim=1)
        elif self.fusion_type == 'avg_pool':
            fused = (t_p + v_p) / 2.0
        elif self.fusion_type == 'element_wise':
            fused = t_p * v_p
        elif self.fusion_type == 'gated':
            concat_feat = torch.cat((t_p, v_p), dim=1)
            gates = self.sigmoid(self.gate(concat_feat))
            gate_t = gates[:, :self.common_dim]
            gate_v = gates[:, self.common_dim:]
            fused = (gate_t * t_p) + (gate_v * v_p)
            
        return self.classifier(fused)

# ==========================================
# 4. Load Model and Weights
# ==========================================
print(f"Loading weights from {MODEL_PATH}...")
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
saved_fusion_type = checkpoint['fusion_type']
print(f"Detected Fusion Strategy used during training: {saved_fusion_type}")

# Initialize raw backbones (the checkpoint will overwrite them with the trained weights)
print("Initializing base architectures...")
text_base = AutoModel.from_pretrained("google/muril-base-cased")
resnet_base = models.resnet50()
image_base = nn.Sequential(*list(resnet_base.children())[:-1])

# Build complete model and load state dictionary
model = MurilResnetFusionNet(text_base, image_base, saved_fusion_type)
model.load_state_dict(checkpoint['state_dict'])
model.to(DEVICE)
model.eval()

# ==========================================
# 5. Prediction Loop
# ==========================================
print("\n[Data] Preparing Test DataLoader...")
# NOTE: Make sure your test dataframe is named 'test_df'
test_loader = DataLoader(MurilResnetTestDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)

file_names = []
predictions = []

print("Running inference...")
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        outputs = model(
            batch['input_ids'].to(DEVICE), 
            batch['attention_mask'].to(DEVICE), 
            batch['image'].to(DEVICE)
        )
        
        # Get the predicted class (0 or 1)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        
        file_names.extend(batch['file_name'])
        predictions.extend(preds)

# ==========================================
# 6. Save Submission File
# ==========================================
submission_df = pd.DataFrame({
    'file_name': file_names,
    'misogynous': predictions
})

submission_df.to_csv(SUBMISSION_PATH, index=False)
print("\n==============================================")
print(f"✅ INFERENCE COMPLETE!")
print(f"📄 Predictions successfully saved to: {SUBMISSION_PATH}")
print("==============================================")

--- Starting Phase 4: Model Inference ---
Using Device: cuda
Loading weights from /kaggle/working/saved_models/best_muril_resnet_fusion.pth...
Detected Fusion Strategy used during training: element_wise
Initializing base architectures...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Running inference...



Predicting: 100%|██████████| 32/32 [00:41<00:00,  1.31s/it]


✅ INFERENCE COMPLETE!
📄 Predictions successfully saved to: submission.csv


new text and image model hatebert and clip


In [12]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import CLIPImageProcessor, CLIPVisionModel
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from PIL import Image
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings("ignore")

print("--- Starting Phase 2: HateBERT & CLIP Unimodal Training ---")

# ==========================================
# 1. Setup & Config
# ==========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {DEVICE}")

BATCH_SIZE = 32
EPOCHS = 3 
LR = 2e-5  # Lower learning rate is better for fine-tuning Transformers
MAX_LEN = 128

# Create directories to save our fine-tuned unimodal weights
SAVE_DIR = '/kaggle/working/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)
TEXT_MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'best_hatebert.pth')
IMAGE_MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'best_clip_vision.pth')

# ==========================================
# 2. HateBERT Text Unimodal
# ==========================================
print("\n" + "="*50)
print("🧠 PART A: TRAINING HATEBERT (TEXT UNIMODAL)")
print("="*50)

# Dataset for Text
class HateBertDataset(Dataset):
    def __init__(self, df, tokenizer_name="GroNLP/hateBERT"):
        self.df = df.reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.loc[idx, 'transcriptions'])
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        label = torch.tensor(self.df.loc[idx, 'misogyny'], dtype=torch.long)
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': label
        }

# Initialize Model & DataLoaders
text_model = AutoModelForSequenceClassification.from_pretrained("GroNLP/hateBERT", num_labels=2).to(DEVICE)
text_train_loader = DataLoader(HateBertDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
text_val_loader = DataLoader(HateBertDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)

optimizer_text = torch.optim.AdamW(text_model.parameters(), lr=LR)
best_text_f1 = 0.0

# Text Training Loop
for epoch in range(EPOCHS):
    text_model.train()
    for batch in tqdm(text_train_loader, desc=f"HateBERT Ep {epoch+1}/{EPOCHS} [Train]", leave=False):
        optimizer_text.zero_grad()
        outputs = text_model(
            input_ids=batch['input_ids'].to(DEVICE), 
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['label'].to(DEVICE)
        )
        loss = outputs.loss
        loss.backward()
        optimizer_text.step()
        
    text_model.eval()
    val_preds, val_labels, val_probs = [], [], []
    with torch.no_grad():
        for batch in text_val_loader:
            labels = batch['label'].numpy()
            outputs = text_model(
                input_ids=batch['input_ids'].to(DEVICE), 
                attention_mask=batch['attention_mask'].to(DEVICE)
            )
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[:, 1]
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            
            val_preds.extend(preds)
            val_labels.extend(labels)
            val_probs.extend(probs)
            
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    val_auc = roc_auc_score(val_labels, val_probs)
    
    print(f"HateBERT Epoch {epoch+1} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f}")
    
    if val_f1 > best_text_f1:
        best_text_f1 = val_f1
        torch.save(text_model.state_dict(), TEXT_MODEL_SAVE_PATH)
        print(f"⭐ Saved new best HateBERT model! (F1: {best_text_f1:.4f})")

# Clean up memory
del text_model, optimizer_text, text_train_loader, text_val_loader
torch.cuda.empty_cache()
gc.collect()


# ==========================================
# 3. CLIP Vision Image Unimodal
# ==========================================
print("\n" + "="*50)
print("🖼️ PART B: TRAINING CLIP VISION (IMAGE UNIMODAL)")
print("="*50)

# Custom CLIP Classifier Architecture
class CLIPImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        # CLIP ViT-Base has a hidden size of 768
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, pixel_values):
        outputs = self.clip_vision(pixel_values=pixel_values)
        # Extract the pooled output (CLS token representation of the image)
        pooled_output = outputs.pooler_output 
        return self.classifier(pooled_output)

# Dataset for Images
class CLIPImageDataset(Dataset):
    def __init__(self, df, processor_name="openai/clip-vit-base-patch32"):
        self.df = df.reset_index(drop=True)
        self.processor = CLIPImageProcessor.from_pretrained(processor_name)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.df.loc[idx, 'image_id']).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), color='black')
            
        # CLIP's processor handles the resizing and normalization perfectly
        pixel_values = self.processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
        label = torch.tensor(self.df.loc[idx, 'misogyny'], dtype=torch.long)
        
        return {
            'pixel_values': pixel_values,
            'label': label
        }

# Initialize Model & DataLoaders
image_model = CLIPImageClassifier().to(DEVICE)
image_train_loader = DataLoader(CLIPImageDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
image_val_loader = DataLoader(CLIPImageDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)

optimizer_image = torch.optim.AdamW(image_model.parameters(), lr=LR)
criterion_image = nn.CrossEntropyLoss()
best_image_f1 = 0.0

# Image Training Loop
for epoch in range(EPOCHS):
    image_model.train()
    for batch in tqdm(image_train_loader, desc=f"CLIP Ep {epoch+1}/{EPOCHS} [Train]", leave=False):
        optimizer_image.zero_grad()
        outputs = image_model(pixel_values=batch['pixel_values'].to(DEVICE))
        loss = criterion_image(outputs, batch['label'].to(DEVICE))
        loss.backward()
        optimizer_image.step()
        
    image_model.eval()
    val_preds, val_labels, val_probs = [], [], []
    with torch.no_grad():
        for batch in image_val_loader:
            labels = batch['label'].numpy()
            outputs = image_model(pixel_values=batch['pixel_values'].to(DEVICE))
            probs = torch.softmax(outputs, dim=1).cpu().numpy()[:, 1]
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            val_preds.extend(preds)
            val_labels.extend(labels)
            val_probs.extend(probs)
            
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    val_auc = roc_auc_score(val_labels, val_probs)
    
    print(f"CLIP Vision Epoch {epoch+1} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f}")
    
    if val_f1 > best_image_f1:
        best_image_f1 = val_f1
        torch.save(image_model.state_dict(), IMAGE_MODEL_SAVE_PATH)
        print(f"⭐ Saved new best CLIP Vision model! (F1: {best_image_f1:.4f})")

print("\n==============================================")
print("🎉 PHASE 2 COMPLETE!")
print(f"Text Model saved to: {TEXT_MODEL_SAVE_PATH}")
print(f"Image Model saved to: {IMAGE_MODEL_SAVE_PATH}")
print("Ready for Phase 3 Fusion!")
print("==============================================")

--- Starting Phase 2: HateBERT & CLIP Unimodal Training ---
Using Device: cuda

🧠 PART A: TRAINING HATEBERT (TEXT UNIMODAL)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


HateBERT Epoch 1 | Acc: 0.7396 | AUC: 0.7659 | F1: 0.6917
⭐ Saved new best HateBERT model! (F1: 0.6917)


HateBERT Epoch 2 | Acc: 0.7382 | AUC: 0.7422 | F1: 0.6753


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight        

CLIP Vision Epoch 1 | Acc: 0.7354 | AUC: 0.7432 | F1: 0.6735
⭐ Saved new best CLIP Vision model! (F1: 0.6735)


CLIP Vision Epoch 2 | Acc: 0.7474 | AUC: 0.7350 | F1: 0.6519


CLIP Vision Epoch 3 | Acc: 0.7311 | AUC: 0.7303 | F1: 0.6678

🎉 PHASE 2 COMPLETE!
Text Model saved to: /kaggle/working/saved_models/best_hatebert.pth
Image Model saved to: /kaggle/working/saved_models/best_clip_vision.pth
Ready for Phase 3 Fusion!
